# 04 — Tối ưu hóa danh mục Markowitz

Notebook tái tạo đường biên hiệu quả long-only và vị trí của Min Variance, Max Sharpe, Equal Weight. Cửa sổ 24 tháng 05/2024–04/2026 chỉ dùng thông tin có trước kỳ triển khai 06/2026.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from prepare_vn100_backtest_inputs import prepare
from run_backtest import read_membership, read_monthly
from src.portfolio.markowitz import estimate, frontier, portfolio_stats, solve

## Chuẩn bị panel

Nếu chưa có input cục bộ, cell dưới chuyển output factor đã version hóa sang định dạng backtest. Thư mục đã tồn tại sẽ được giữ nguyên.

In [ ]:
INPUT = ROOT / 'local_runs' / 'notebook_vn100_inputs'
if not INPUT.exists():
    prepare(ROOT / 'outputs' / 'vn_period_factors', INPUT)

returns = read_monthly(INPUT / 'returns.csv', ['date', 'ticker', 'return'], allow_sparse=True)
membership = read_membership(INPUT / 'membership.csv')
rf = read_monthly(INPUT / 'risk_free.csv', ['date', 'rf_return'])

train_start = pd.Timestamp('2024-05-31')
train_end = pd.Timestamp('2026-04-30')
deployment = pd.Timestamp('2026-06-30')
history = returns.loc[train_start:train_end]
eligible = membership.loc[deployment] & history.notna().all(axis=0)
history = history.loc[:, eligible]
assert len(history) == 24 and history.shape[1] >= 2
pd.Series({'training_months': len(history), 'eligible_assets': history.shape[1], 'deployment': deployment.date()})

In [ ]:
mu, covariance = estimate(history, ridge=1e-8)
frontier_points, frontier_weights = frontier(mu, covariance, points=50)
monthly_rf = float(rf.reindex(history.index).mean())
portfolios = {
    'MinVariance': solve(mu, covariance, 'min_variance'),
    'MaxSharpe': solve(mu, covariance, 'max_sharpe', rf=monthly_rf),
    'EqualWeight': pd.Series(1 / len(mu), index=mu.index),
}
stats = pd.DataFrame([
    {'portfolio': name, **portfolio_stats(weights, mu, covariance, monthly_rf)}
    for name, weights in portfolios.items()
]).set_index('portfolio')
display(stats)
display(pd.DataFrame(portfolios).sort_values('MaxSharpe', ascending=False).head(15))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(frontier_points['volatility_monthly'] * 100, frontier_points['expected_return_monthly'] * 100, linewidth=2, label='Đường biên hiệu quả')
for name, row in stats.iterrows():
    x, y = row['volatility_monthly'] * 100, row['expected_return_monthly'] * 100
    ax.scatter(x, y, s=70, zorder=3)
    ax.annotate(name, (x, y), xytext=(6, 6), textcoords='offset points')
ax.set(title='Đường biên hiệu quả long-only', xlabel='Độ lệch chuẩn tháng (%)', ylabel='Lợi suất kỳ vọng tháng (%)')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

Kết quả trên là ước lượng trong mẫu của một cửa sổ huấn luyện, không phải hiệu suất backtest. Notebook 05 mới đánh giá ngoài mẫu theo walk-forward và trừ phí giao dịch.